# Optuna Study with PatchCore and PDT Dataset to Collect Logs 

In [1]:
import torch

torch.cuda.set_device(1)                 # make GPU 1 the default for this process
DEVICE = torch.device("cuda:1")          # always use this explicitly
torch.cuda.set_per_process_memory_fraction(0.7, device=1)

print("device_count:", torch.cuda.device_count())
print("current_device:", torch.cuda.current_device())
print("using:", DEVICE)


device_count: 2
current_device: 1
using: cuda:1


In [14]:
# =========================
# 0) Setup: imports & config
# =========================
from __future__ import annotations

from pathlib import Path
import random
import json

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from PIL import Image, ImageFilter

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from sklearn.neighbors import NearestNeighbors



SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


device: cuda


In [3]:
# =========================
# 1) Paths: locate ROOT and dataset folders
# =========================
ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "fast").is_dir() and (p / "shared").is_dir()
)

DATASET_DIR = ROOT / "shared" / "pdt" / "dataset_public"
TRAIN_GOOD_DIR = DATASET_DIR / "train" / "good"
TRAIN_BAD_DIR  = DATASET_DIR / "train" / "bad"   # ignored for anomaly training
TEST_GOOD_DIR  = DATASET_DIR / "test" / "good"
TEST_BAD_DIR   = DATASET_DIR / "test" / "bad"

for p in [TRAIN_GOOD_DIR, TEST_GOOD_DIR, TEST_BAD_DIR]:
    assert p.is_dir(), f"Missing directory: {p}"

print("ROOT:", ROOT)
print("DATASET_DIR:", DATASET_DIR)


ROOT: /home/jovyan
DATASET_DIR: /home/jovyan/shared/pdt/dataset_public


### Index dataset: build dataframes

In [4]:
# =========================
# 2) Index images + split test -> val/final
# =========================
def list_bmps(folder: Path) -> list[Path]:
    paths = list(folder.rglob("*.bmp")) + list(folder.rglob("*.BMP"))
    return sorted(set(paths))

train_good_paths = list_bmps(TRAIN_GOOD_DIR)
train_bad_paths  = list_bmps(TRAIN_BAD_DIR) if TRAIN_BAD_DIR.is_dir() else []  # ignored
test_good_paths  = list_bmps(TEST_GOOD_DIR)
test_bad_paths   = list_bmps(TEST_BAD_DIR)

df_train_good = pd.DataFrame({"path": [str(p) for p in train_good_paths], "label": 0})
df_test = pd.DataFrame({
    "path":  [str(p) for p in (test_good_paths + test_bad_paths)],
    "label": [0]*len(test_good_paths) + [1]*len(test_bad_paths),
})

df_val, df_final = train_test_split(
    df_test,
    test_size=0.5,
    random_state=SEED,
    stratify=df_test["label"],
)
df_val = df_val.reset_index(drop=True)
df_final = df_final.reset_index(drop=True)

print("train_good:", len(df_train_good))
print("train_bad (ignored):", len(train_bad_paths))
print("val:", df_val["label"].value_counts().to_dict())
print("final:", df_final["label"].value_counts().to_dict())


train_good: 4538
train_bad (ignored): 2390
val: {0: 567, 1: 300}
final: {0: 568, 1: 299}


In [15]:
# =========================
# 2.5) Export fixed splits for downstream evaluation
# =========================

SPLIT_OUT = ROOT / "fast" / "x_peer_pdt_hyperoptxai" / "hpo_pdt_dataset" / "patchcore_splits"
SPLIT_OUT.mkdir(parents=True, exist_ok=True)

df_train_good.to_csv(SPLIT_OUT / "df_train_good.csv", index=False)
df_val.to_csv(SPLIT_OUT / "df_val.csv", index=False)
df_final.to_csv(SPLIT_OUT / "df_final.csv", index=False)

split_meta = {
    "seed": SEED,
    "dataset_dir": str(DATASET_DIR),
    "train_good_dir": str(TRAIN_GOOD_DIR),
    "test_good_dir": str(TEST_GOOD_DIR),
    "test_bad_dir": str(TEST_BAD_DIR),
    "val_final_split": {
        "source": "df_test",
        "method": "train_test_split(test_size=0.5, stratify=label)",
    },
    "counts": {
        "train_good": int(len(df_train_good)),
        "val": int(len(df_val)),
        "final": int(len(df_final)),
        "val_labels": df_val["label"].value_counts().to_dict(),
        "final_labels": df_final["label"].value_counts().to_dict(),
    },
}

with open(SPLIT_OUT / "split_meta.json", "w", encoding="utf-8") as f:
    json.dump(split_meta, f, indent=2)

print("Exported splits to:", SPLIT_OUT)


Exported splits to: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/patchcore_splits


In [5]:
# =========================
# 3) Soft params (separate): effort, quality, review budget
# =========================
SOFT_TRAIN_FRACTION_RANGE = (0.2, 1.0)
SOFT_CORRUPTION_LEVELS = ["none", "mild", "strong"]
SOFT_REVIEW_BUDGETS = [i / 1000 for i in range(5, 501, 5)]

def _stable_int_seed(text: str, seed: int) -> int:
    return (zlib.crc32(text.encode("utf-8")) + seed) & 0xFFFFFFFF


def corrupt_pil(img: Image.Image, level: str, seed_int: int) -> Image.Image:
    """Deterministic corruption applied to train/good only."""
    if level == "none":
        return img

    rng = np.random.default_rng(seed_int)
    out = img

    if level in {"mild", "strong"}:
        radius = 0.6 if level == "mild" else 1.2
        out = out.filter(ImageFilter.GaussianBlur(radius=radius))

    if level == "strong":
        arr = np.asarray(out).astype(np.float32)
        noise_std = 8.0
        noise = rng.normal(0.0, noise_std, size=arr.shape).astype(np.float32)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        out = Image.fromarray(arr)

    return out

# Helper: quantize any fraction to 1% steps
def q01(x: float, lo: float, hi: float) -> float:
    """Quantize to 1% steps and clip to [lo, hi]."""
    x = max(lo, min(hi, x))
    return round(x * 100.0) / 100.0


### Data loading

In [6]:
# =========================
# 4) Dataset + loaders (preprocess includes resize + optional center-crop)
# =========================
class ImagePathDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        preproc: transforms.Compose,
        corruption_level: str = "none",
        apply_corruption: bool = False,
    ):
        self.paths = df["path"].tolist()
        self.labels = df["label"].to_numpy().astype(int)
        self.preproc = preproc
        self.corruption_level = corruption_level
        self.apply_corruption = apply_corruption

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")

        if self.apply_corruption and self.corruption_level != "none":
            s = _stable_int_seed(path, SEED)
            img = corrupt_pil(img, self.corruption_level, s)

        x = self.preproc(img)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def make_preproc(image_size: tuple[int, int], center_crop_size: tuple[int, int] | None):
    t = [transforms.Resize(image_size)]
    if center_crop_size is not None:
        t.append(transforms.CenterCrop(center_crop_size))
    t += [
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ]
    return transforms.Compose(t)


def make_loaders(
    df_train_good_sub: pd.DataFrame,
    image_size: tuple[int, int],
    center_crop_size: tuple[int, int] | None,
    batch_size: int,
    corruption_level: str,
):
    preproc = make_preproc(image_size=image_size, center_crop_size=center_crop_size)

    train_ds = ImagePathDataset(df_train_good_sub, preproc, corruption_level=corruption_level, apply_corruption=True)
    val_ds   = ImagePathDataset(df_val,          preproc, corruption_level="none", apply_corruption=False)
    final_ds = ImagePathDataset(df_final,        preproc, corruption_level="none", apply_corruption=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    final_loader = DataLoader(final_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, final_loader


### PatchCore core: feature extractor + embedding

In [7]:
# =========================
# 5) PatchCore core: feature extractor + patch embeddings + memory bank + scoring
# =========================
def build_backbone(backbone: str, pre_trained: bool) -> torch.nn.Module:
    weights = "DEFAULT" if pre_trained else None

    if backbone == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pre_trained else None)
    elif backbone == "resnet34":
        m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT if pre_trained else None)
    elif backbone == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pre_trained else None)
    elif backbone == "wide_resnet50_2":
        m = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.DEFAULT if pre_trained else None)
    else:
        raise ValueError(f"Unsupported backbone: {backbone}")

    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    return m


class FeatureExtractor(torch.nn.Module):
    def __init__(self, backbone: str, layers: tuple[str, ...], pre_trained: bool):
        super().__init__()
        self.backbone = build_backbone(backbone, pre_trained)
        self.layers = layers
        self._features: dict[str, torch.Tensor] = {}

        named = dict(self.backbone.named_modules())
        for name in layers:
            if name not in named:
                raise ValueError(f"Layer '{name}' not found in backbone '{backbone}'")
            named[name].register_forward_hook(self._hook(name))

    def _hook(self, name: str):
        def fn(module, inp, out):
            self._features[name] = out
        return fn

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        _ = self.backbone(x)
        return [self._features[name] for name in self.layers]


def to_patch_embeddings(
    feat_maps: list[torch.Tensor],
    max_patches_per_image: int | None = None,
    seed: int = 0,
) -> torch.Tensor:
    H = max(f.shape[-2] for f in feat_maps)
    W = max(f.shape[-1] for f in feat_maps)

    resized = []
    for f in feat_maps:
        if (f.shape[-2], f.shape[-1]) != (H, W):
            f = torch.nn.functional.interpolate(f, size=(H, W), mode="bilinear", align_corners=False)
        resized.append(f)

    f_cat = torch.cat(resized, dim=1)          # [B, D, H, W]
    B, D, _, _ = f_cat.shape

    f_flat = f_cat.flatten(2).transpose(1, 2).contiguous()  # [B, P, D]
    P = f_flat.shape[1]

    if max_patches_per_image is not None and max_patches_per_image < P:
        g = torch.Generator(device=f_flat.device)
        g.manual_seed(seed)
        idx = torch.randperm(P, generator=g, device=f_flat.device)[:max_patches_per_image]
        f_flat = f_flat[:, idx, :]  # [B, P', D]

    return f_flat.reshape(-1, D)


def coreset_sample(embeddings: np.ndarray, ratio: float, seed: int) -> np.ndarray:
    n = embeddings.shape[0]
    m = max(1, int(n * ratio))
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=m, replace=False)
    return embeddings[idx]


@torch.no_grad()
def fit_memory_bank(
    extractor: FeatureExtractor,
    train_loader: DataLoader,
    coreset_sampling_ratio: float,
    max_patches_per_image: int,
) -> np.ndarray:
    extractor = extractor.to(DEVICE).eval()

    kept = []
    for x, _ in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)

        patches = to_patch_embeddings(
            feats,
            max_patches_per_image=max_patches_per_image,
            seed=SEED,
        )
        kept.append(patches.detach().cpu().numpy())

        del x, feats, patches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    emb = np.vstack(kept)
    return coreset_sample(emb, ratio=coreset_sampling_ratio, seed=SEED)


@torch.no_grad()
def score_images(
    extractor: FeatureExtractor,
    loader: DataLoader,
    memory_bank: np.ndarray,
    num_neighbors: int,
    reduction: str,
    max_patches_per_image: int,
) -> tuple[np.ndarray, np.ndarray]:
    extractor = extractor.to(DEVICE).eval()

    nn = NearestNeighbors(n_neighbors=num_neighbors, algorithm="auto")
    nn.fit(memory_bank)

    y_true_list, y_score_list = [], []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)

        patches = to_patch_embeddings(
            feats,
            max_patches_per_image=max_patches_per_image,
            seed=SEED,
        ).detach().cpu().numpy()

        dists, _ = nn.kneighbors(patches, return_distance=True)
        patch_scores = dists.mean(axis=1)

        B = y.shape[0]
        P = patch_scores.shape[0] // B
        patch_scores = patch_scores.reshape(B, P)

        img_scores = patch_scores.max(axis=1) if reduction == "max" else patch_scores.mean(axis=1)

        y_true_list.append(y.numpy())
        y_score_list.append(img_scores)

        del x, feats, patches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return np.concatenate(y_true_list), np.concatenate(y_score_list)


### Fit + score + metrics (reusable)

In [8]:
# =========================
# 6) Review-budget threshold + F1 objective
# =========================
def threshold_from_review_budget(y_score: np.ndarray, budget: float) -> float:
    """Threshold that flags the top 'budget' fraction (highest scores)."""
    assert 0.0 < budget < 1.0
    n = y_score.shape[0]
    k = max(1, int(np.ceil(budget * n)))
    thr = np.partition(y_score, -k)[-k]
    return float(thr)


def f1_at_review_budget(y_true: np.ndarray, y_score: np.ndarray, budget: float) -> tuple[float, float, float, float]:
    thr = threshold_from_review_budget(y_score, budget)
    y_pred = (y_score >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return float(thr), float(p), float(r), float(f1)


### Optuna: tune PatchCore hyperparameters on validation set

In [9]:
# =========================
# 7) Optuna objective (maximize F1 on val at review budget)
# =========================

LAYERS_MAP = {
    "l2_l3": ("layer2", "layer3"),
    "l1_l2_l3": ("layer1", "layer2", "layer3"),
    "l3": ("layer3",),
    "l2": ("layer2",),
}

IMAGE_SIZE_MAP = {
    "224": (224, 224),
    "256": (256, 256),
    "320": (320, 320),
}

def objective(trial: optuna.Trial) -> float:
    t0 = time.time()

    # ---- soft params (separate) ----
    soft_train_fraction = trial.suggest_float(
        "soft_train_fraction",
        SOFT_TRAIN_FRACTION_RANGE[0],
        SOFT_TRAIN_FRACTION_RANGE[1],
        step=0.01,
    )

    soft_corruption_level = trial.suggest_categorical("soft_corruption_level", SOFT_CORRUPTION_LEVELS)
    soft_review_budget = trial.suggest_categorical("soft_review_budget", SOFT_REVIEW_BUDGETS)

    # ---- model params (PatchCore) ----
    backbone = trial.suggest_categorical("backbone", ["wide_resnet50_2", "resnet18", "resnet34", "resnet50"])

    layers_key = trial.suggest_categorical("layers_key", list(LAYERS_MAP.keys()))
    layers = LAYERS_MAP[layers_key]

    pre_trained = trial.suggest_categorical("pre_trained", [True, False])

    image_size_key = trial.suggest_categorical("image_size_key", list(IMAGE_SIZE_MAP.keys()))
    image_size = IMAGE_SIZE_MAP[image_size_key]

    center_crop_key = trial.suggest_categorical("center_crop_key", ["none", "0.875"])
    if center_crop_key == "none":
        center_crop_size = None
    elif center_crop_key == "0.875":
        s = image_size[0]
        center_crop_size = (int(0.875 * s), int(0.875 * s))
    else:
        raise ValueError(center_crop_key)

    coreset_sampling_ratio = trial.suggest_float(
        "coreset_sampling_ratio",
        0.001,
        0.101,     # divisible with step=0.002
        step=0.002,
    )
    num_neighbors = trial.suggest_int("num_neighbors", 1, 30)
    reduction = trial.suggest_categorical("reduction", ["max", "mean"])

    # memory-safety (model-side, not a soft param)
    max_patches_per_image = trial.suggest_categorical("max_patches_per_image", [128, 256, 512, 1024])

    # keep conservative to avoid OOM
    batch_size = trial.suggest_categorical("batch_size", [8, 16])

    # ---- apply soft training effort (diverse but reproducible across trials) ----
    if soft_train_fraction >= 1.0:
        df_sub = df_train_good
    else:
        df_sub = df_train_good.sample(frac=soft_train_fraction, random_state=SEED + trial.number)

    # ---- loaders + model ----
    train_loader, val_loader, _ = make_loaders(
        df_train_good_sub=df_sub,
        image_size=image_size,
        center_crop_size=center_crop_size,
        batch_size=batch_size,
        corruption_level=soft_corruption_level,
    )

    extractor = FeatureExtractor(backbone=backbone, layers=layers, pre_trained=pre_trained)

    # ---- run trial  ----
    memory_bank = fit_memory_bank(
        extractor,
        train_loader,
        coreset_sampling_ratio=coreset_sampling_ratio,
        max_patches_per_image=max_patches_per_image,
    )

    y_true_val, y_score_val = score_images(
        extractor,
        val_loader,
        memory_bank,
        num_neighbors=num_neighbors,
        reduction=reduction,
        max_patches_per_image=max_patches_per_image,
    )

    thr, p, r, f1 = f1_at_review_budget(y_true_val, y_score_val, soft_review_budget)
    auroc = roc_auc_score(y_true_val, y_score_val)

    # optional but useful: log the *effective* discretization for later analysis
    trial.set_user_attr("soft_train_fraction_q01", float(soft_train_fraction))
    trial.set_user_attr("memory_bank_size", int(getattr(memory_bank, "shape", [len(memory_bank)])[0]))

    runtime_sec = time.time() - t0

    trial.set_user_attr("threshold_val_budget", float(thr))
    trial.set_user_attr("precision_val_at_budget", float(p))
    trial.set_user_attr("recall_val_at_budget", float(r))
    trial.set_user_attr("f1_val_at_budget", float(f1))
    trial.set_user_attr("auroc_val", float(auroc))
    trial.set_user_attr("runtime_sec", float(runtime_sec))
    trial.set_user_attr("train_good_used", int(len(df_sub)))
    trial.set_user_attr("oom", False)

    return float(f1)

In [11]:
# =========================
# 8) Persistent study runner (resume to TARGET_TRIALS)
# =========================
DATASET_NAME = "pdt_public_patchcore"
TARGET_TRIALS = 3000

OUT_DIR = ROOT / "fast" / "x_peer_pdt_hyperoptxai" / "hpo_pdt_dataset" / "outputs_patchcore"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = OUT_DIR / f"{DATASET_NAME}_hpo_logs.db"
STORAGE = f"sqlite:///{DB_PATH.as_posix()}"

sampler = optuna.samplers.TPESampler(
    seed=SEED,
    n_startup_trials=1500,
    multivariate=False,
    group=False,
)


study = optuna.create_study(
    direction="maximize",
    study_name=f"{DATASET_NAME}_hpo_logs_v1.0.1",
    storage=STORAGE,
    load_if_exists=True,
    sampler=sampler,
)

already_done = len(study.trials)
remaining = max(TARGET_TRIALS - already_done, 0)

if remaining:
    print(f"Running {remaining} additional trials (have {already_done}, need {TARGET_TRIALS})")
    study.optimize(objective, n_trials=remaining, gc_after_trial=True)
else:
    print(f"Target of {TARGET_TRIALS} trials already reached – nothing to do")

print("Best F1 (val @ budget):", study.best_value)
print("Best params:", study.best_params)
print("DB:", DB_PATH)


[I 2026-01-30 22:38:36,501] Using an existing study with name 'pdt_public_patchcore_hpo_logs_v1.0.1' instead of creating a new one.


Running 2754 additional trials (have 246, need 3000)


[I 2026-01-30 22:39:06,779] Trial 246 finished with value: 0.6882453151618398 and parameters: {'soft_train_fraction': 0.5, 'soft_corruption_level': 'none', 'soft_review_budget': 0.33, 'backbone': 'wide_resnet50_2', 'layers_key': 'l3', 'pre_trained': True, 'image_size_key': '256', 'center_crop_key': '0.875', 'coreset_sampling_ratio': 0.055, 'num_neighbors': 25, 'reduction': 'max', 'max_patches_per_image': 1024, 'batch_size': 8}. Best is trial 207 with value: 0.8517350157728707.
[I 2026-01-30 22:39:22,731] Trial 247 finished with value: 0.08585858585858586 and parameters: {'soft_train_fraction': 0.6100000000000001, 'soft_corruption_level': 'none', 'soft_review_budget': 0.11, 'backbone': 'resnet34', 'layers_key': 'l2', 'pre_trained': True, 'image_size_key': '320', 'center_crop_key': '0.875', 'coreset_sampling_ratio': 0.099, 'num_neighbors': 26, 'reduction': 'mean', 'max_patches_per_image': 128, 'batch_size': 8}. Best is trial 207 with value: 0.8517350157728707.
[I 2026-01-30 22:40:05,356]

Best F1 (val @ budget): 0.907563025210084
Best params: {'soft_train_fraction': 0.9000000000000001, 'soft_corruption_level': 'none', 'soft_review_budget': 0.34, 'backbone': 'resnet34', 'layers_key': 'l3', 'pre_trained': True, 'image_size_key': '224', 'center_crop_key': '0.875', 'coreset_sampling_ratio': 0.091, 'num_neighbors': 1, 'reduction': 'mean', 'max_patches_per_image': 1024, 'batch_size': 16}
DB: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/outputs_patchcore/pdt_public_patchcore_hpo_logs.db
